# Alma — Random Forest exploration
PyCharm / Jupyter notebook · CS5785 Human–AI Text

## Team notes (Kaggle.docx, reviewed 2026-11-07)
- Alma: Random Forest.
- Hold out **2,000** labeled examples with **random_state=42**. The assignment PDF requires stratification.
- Record **train and validation AUROC**.
- Other proposed methods: TF-IDF + Logistic Regression; TF-IDF + BERT/MiniLM embeddings → Logistic Regression (Abishek); term-frequency SVM and embeddings (Jennie); Naive Bayes; MiniLM + MLP.
- The feature method for Random Forest and the exact shared validation IDs are **not finalized**.

This is an exploration notebook, not completed competition results. The downloaded dataset has now been checked and connected. Training results have not yet been computed.
Keep the original row order. Do not choose a split because it produces a higher score.
The official unlabeled Kaggle test file is separate from this validation holdout.

## Assignment data (PDF pages 1–2)
- train.csv: 10,000 labeled passages; human=0 (5,000), AI=1 (5,000).
- Hold out a fixed stratified 2,000 rows (1,000 per class), leaving 8,000 training rows.
- unlabeled.csv: 3,000 unlabeled passages for the required unlabeled-data experiment.
- test.csv: 3,000 unlabeled passages for final Kaggle probabilities.
- train_starter_2000.csv: 2,000-row subset of train.csv for quick experiments; not extra data or an automatically designated validation set.
Verified CSV columns: labeled files use id, text, label; test/unlabeled use id, text. sample_submission.csv has id, label; label=0.5 is a placeholder probability, not ground truth.


## 1. Environment
Open this .ipynb in PyCharm and select a Python interpreter/kernel. Run the installation cell if imports are missing. MiniLM is optional and downloads pretrained weights on first use; verify competition rules before enabling it.

In [ ]:
# Uncomment and run once in the selected notebook kernel:
# %pip install pandas numpy scikit-learn ipykernel
# Optional, only for MiniLM:
# %pip install sentence-transformers

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import time
import sys

import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

print("Python:", sys.version)
print("scikit-learn:", sklearn.__version__)

## 2. Configuration — connected to downloaded CSVs
Paths and column names below were verified against the downloaded files.
The assignment PDF (page 2) requires a fixed stratified 2,000-row holdout. STRATIFY=True is therefore required; coordinate the actual sample IDs with teammates.
The PDF confirms human=0 and AI=1, so POSITIVE_LABEL=1.

In [ ]:
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
TRAIN_CSV = DATA_DIR / "train.csv"
UNLABELED_CSV = DATA_DIR / "unlabeled.csv"
TEST_CSV = DATA_DIR / "test.csv"
STARTER_CSV = DATA_DIR / "train_starter_2000.csv"
SAMPLE_SUBMISSION_CSV = DATA_DIR / "sample_submission.csv"
TEXT_COL = "text"       # Verified against downloaded CSVs
LABEL_COL = "label"     # Verified: human=0, AI=1
ID_COL = "id"          # Verified unique IDs; use these for shared splits
POSITIVE_LABEL = 1     # PDF confirms: human=0, AI=1

VALIDATION_SIZE = 2000
SPLIT_SEED = 42
STRATIFY = True         # Required by assignment PDF, page 2
SHARED_SPLIT_CSV = None # Later: Path to a teammate's exported split CSV
RUN_MINILM = False
OUTPUT_DIR = PROJECT_DIR / "outputs"

In [ ]:
if not TRAIN_CSV.exists():
    raise FileNotFoundError(
        f"Add the labeled training CSV at {TRAIN_CSV}, or update TRAIN_CSV. "
        "Do not use the official unlabeled test CSV."
    )
df = pd.read_csv(TRAIN_CSV)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head(3))

In [ ]:
required = [TEXT_COL, LABEL_COL] + ([ID_COL] if ID_COL else [])
assert all(c in df.columns for c in required), "Update the column settings."
assert len(df) > VALIDATION_SIZE, "Need more than 2,000 labeled rows."
assert df[LABEL_COL].notna().all(), "Resolve missing labels before splitting."
assert df[LABEL_COL].nunique() == 2, "This notebook expects binary classification."
assert POSITIVE_LABEL in df[LABEL_COL].unique(), "Check POSITIVE_LABEL."

# Preserve row order; do not silently drop duplicates or missing text.
texts = df[TEXT_COL].fillna("").astype(str)
y = (df[LABEL_COL] == POSITIVE_LABEL).astype(int).to_numpy()
ids = df[ID_COL].astype(str) if ID_COL else pd.Series(
    [f"row_{i}" for i in range(len(df))], index=df.index
)
assert ids.is_unique, "Sample IDs must be unique."
if ID_COL:
    assert df[ID_COL].notna().all(), "Sample IDs cannot be missing."

dataset_sha256 = hashlib.sha256(TRAIN_CSV.read_bytes()).hexdigest()
print("Dataset SHA256:", dataset_sha256)
print("Missing text:", int(df[TEXT_COL].isna().sum()))
print("Exact repeated text rows:", int(texts.duplicated().sum()))
display(df[LABEL_COL].value_counts(dropna=False))
# If texts share an author/source/document, discuss group splitting before final evaluation.

## 3. Provisional holdout and reusable split manifest
A shared manifest identifies actual samples, not just a seed. This loader requires the same CSV bytes and sample IDs to prevent accidental mismatches. Row-based IDs are valid only for that exact file.
No data are removed or reordered before the split.

In [ ]:
if SHARED_SPLIT_CSV is None:
    train_idx, val_idx = train_test_split(
        np.arange(len(df)),
        test_size=VALIDATION_SIZE,
        random_state=SPLIT_SEED,
        shuffle=True,
        stratify=y if STRATIFY else None,
    )
    split_source = "provisional_seed42"
else:
    shared = pd.read_csv(SHARED_SPLIT_CSV, dtype={"sample_id": str})
    assert {"sample_id", "split", "dataset_sha256"}.issubset(shared.columns)
    assert len(shared) == len(df) and shared["sample_id"].is_unique
    assert set(shared["sample_id"]) == set(ids)
    assert shared["dataset_sha256"].eq(dataset_sha256).all(), "Dataset differs."
    assert set(shared["split"]) == {"train", "validation"}
    assignment = shared.set_index("sample_id").loc[ids, "split"].to_numpy()
    train_idx = np.flatnonzero(assignment == "train")
    val_idx = np.flatnonzero(assignment == "validation")
    split_source = str(SHARED_SPLIT_CSV)

assert len(val_idx) == VALIDATION_SIZE
assert not set(train_idx) & set(val_idx)
assert len(train_idx) + len(val_idx) == len(df)
assert len(np.unique(y[train_idx])) == len(np.unique(y[val_idx])) == 2
X_train, X_val = texts.iloc[train_idx], texts.iloc[val_idx]
y_train, y_val = y[train_idx], y[val_idx]

split_manifest = pd.DataFrame({
    "sample_id": ids.to_numpy(),
    "split": np.where(np.isin(np.arange(len(df)), val_idx), "validation", "train"),
    "dataset_sha256": dataset_sha256,
})
split_id = hashlib.sha256(
    split_manifest.sort_values("sample_id").to_csv(index=False).encode()
).hexdigest()[:16]
overlap = set(X_train) & set(X_val)
print("Split ID:", split_id, "| Source:", split_source)
print("Train:", len(train_idx), "| Validation:", len(val_idx))
print("Positive fraction:", y_train.mean(), y_val.mean())
print("Distinct identical texts crossing the split:", len(overlap))
if overlap:
    print("Review duplicate leakage with the team before treating scores as final.")

## 4. Compare representations with the same Random Forest
Start with TF-IDF and term frequency (raw counts). These are candidate methods, not team decisions.
Fit vocabulary and IDF on training text only. Keep sparse matrices sparse.
MiniLM, when enabled, is a frozen pretrained encoder; no fine-tuning is performed.

AUROC uses positive-class **probabilities**, not predicted 0/1 labels. A large train–validation gap suggests overfitting. Train AUROC is an in-sample diagnostic.

In [ ]:
RF_PARAMS = dict(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1,
)
vectorizers = {
    "tfidf_word_1_2": TfidfVectorizer(
        max_features=20000, ngram_range=(1, 2), min_df=2, dtype=np.float32
    ),
    "count_word_1_2": CountVectorizer(
        max_features=20000, ngram_range=(1, 2), min_df=2, dtype=np.float32
    ),
}
results = []
predictions = pd.DataFrame({
    "sample_id": ids.iloc[val_idx].to_numpy(),
    "y_true": y_val,
})
fitted = {}

def evaluate_rf(name, train_features, val_features, feature_config):
    started = time.perf_counter()
    model = RandomForestClassifier(**RF_PARAMS)
    model.fit(train_features, y_train)
    positive_index = list(model.classes_).index(1)
    train_probability = model.predict_proba(train_features)[:, positive_index]
    val_probability = model.predict_proba(val_features)[:, positive_index]
    train_auc = roc_auc_score(y_train, train_probability)
    val_auc = roc_auc_score(y_val, val_probability)
    results.append({
        "representation": name,
        "train_auroc": train_auc,
        "val_auroc": val_auc,
        "train_val_gap": train_auc - val_auc,
        "n_features": train_features.shape[1],
        "rf_fit_predict_seconds": time.perf_counter() - started,
        "split_id": split_id,
        "feature_config": json.dumps(feature_config, default=str, sort_keys=True),
        "rf_params": json.dumps(RF_PARAMS, sort_keys=True),
    })
    predictions[name] = val_probability
    fitted[name] = model
    print(f"{name}: train AUROC={train_auc:.4f}; val AUROC={val_auc:.4f}")

for name, vectorizer in vectorizers.items():
    train_features = vectorizer.fit_transform(X_train)
    val_features = vectorizer.transform(X_val)
    evaluate_rf(name, train_features, val_features, vectorizer.get_params())

In [ ]:
MINILM_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
if RUN_MINILM:
    from sentence_transformers import SentenceTransformer
    encoder = SentenceTransformer(MINILM_MODEL)
    print("Encoder max sequence length:", encoder.max_seq_length)
    # Long documents are truncated by this model; investigate chunking separately.
    train_embeddings = encoder.encode(
        X_train.tolist(), batch_size=32, show_progress_bar=True,
        normalize_embeddings=True, convert_to_numpy=True,
    )
    val_embeddings = encoder.encode(
        X_val.tolist(), batch_size=32, show_progress_bar=True,
        normalize_embeddings=True, convert_to_numpy=True,
    )
    evaluate_rf("minilm_frozen", train_embeddings, val_embeddings, {
        "model": MINILM_MODEL, "normalize_embeddings": True,
        "max_seq_length": encoder.max_seq_length,
    })
else:
    print("MiniLM skipped. Set RUN_MINILM=True only if wanted and permitted.")

In [ ]:
results_df = pd.DataFrame(results).sort_values("val_auroc", ascending=False)
display(results_df[[
    "representation", "train_auroc", "val_auroc", "train_val_gap",
    "n_features", "rf_fit_predict_seconds", "split_id",
]])

## 5. Save this run for comparison
Each run creates a new directory. The CSV manifest can later be shared with teammates.
These are local exploration scores, not Kaggle test scores. Repeatedly tuning on the same validation set can overfit it; use training-only cross-validation for larger searches.
Do not change the split to improve reported AUROC.

In [ ]:
run_dir = OUTPUT_DIR / datetime.now(timezone.utc).strftime("rf_%Y%m%dT%H%M%S_%fZ")
run_dir.mkdir(parents=True, exist_ok=False)
split_manifest.to_csv(run_dir / "split_manifest.csv", index=False)
results_df.to_csv(run_dir / "rf_results.csv", index=False)
predictions.to_csv(run_dir / "validation_predictions.csv", index=False)
metadata = {
    "dataset_sha256": dataset_sha256, "split_id": split_id,
    "split_source": split_source, "validation_size": VALIDATION_SIZE,
    "candidate_split_seed": SPLIT_SEED, "candidate_stratify": STRATIFY,
    "text_column": TEXT_COL, "label_column": LABEL_COL, "id_column": ID_COL,
    "positive_label": POSITIVE_LABEL, "rf_params": RF_PARAMS,
    "sklearn_version": sklearn.__version__, "pandas_version": pd.__version__,
    "numpy_version": np.__version__, "python_version": sys.version,
}
(run_dir / "run_metadata.json").write_text(
    json.dumps(metadata, indent=2, default=str), encoding="utf-8"
)
print("Saved:", run_dir)

## 6. Notes for the report
Fill after running:
- Dataset / positive label:
- Shared or provisional split ID:
- Best RF representation and train / validation AUROC:
- Overfitting gap:
- Duplicate/source leakage concerns:
- Next controlled experiment:

References:
- https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html
- https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html

Next: agree on the shared split manifest; compare methods on those same samples; freeze the method before fitting on all labeled data and generating official test predictions.